# Obligatorio Steam: modelado de datos

En este notebook se construye el modelo estrella a partir de las tablas limpias de la zona refinada (`/rfn/steam/refinado`). Las tablas del modelo se guardan en formato Parquet en la zona de modelos (`/models/steam`), y sobre ellas se crean después las tablas Hive.

## 1. Lectura de la zona refinada

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("steam_modelado").getOrCreate()

REFINADO = "/rfn/steam/refinado"
MODELO = "/models/steam"

juegos = spark.read.parquet(f"{REFINADO}/juegos")
juego_genero = spark.read.parquet(f"{REFINADO}/juego_genero")
usuarios = spark.read.parquet(f"{REFINADO}/usuarios")
usuario_juego = spark.read.parquet(f"{REFINADO}/usuario_juego")
resenas = spark.read.parquet(f"{REFINADO}/resenas")

for nombre, df in [("juegos", juegos), ("juego_genero", juego_genero), ("usuarios", usuarios),
                   ("usuario_juego", usuario_juego), ("resenas", resenas)]:
    print(f"{nombre:15s} {df.count():>10,} filas")

juegos              32,132 filas
juego_genero        71,551 filas
usuarios            87,626 filas
usuario_juego    5,094,082 filas
resenas             58,431 filas


## 2. Conexión con Hive

Antes de construir el modelo comprobamos que la sesión de Spark está conectada al metastore de Hive, donde se van a registrar las tablas del modelo.

In [5]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
| big_data|
|  default|
+---------+



## 3. Dimensiones

### 3.1. dim_juego

Una fila por juego del catálogo, con los atributos que usan las preguntas. Se agrega `rango_precio`, que agrupa los precios en tramos para la pregunta 2. El desarrollador no va en esta tabla porque un juego puede tener varios; se modela aparte en la sección 3.4.

In [6]:
dim_juego = juegos.select(
    "game_id",
    "nombre",
    "publisher",
    "anio_lanzamiento",
    "precio",
    "es_gratuito",
    F.when(F.col("precio").isNull(), "Desconocido")
     .when(F.col("precio") == 0, "Gratis")
     .when(F.col("precio") <= 5, "Hasta $5")
     .when(F.col("precio") <= 15, "$5 a $15")
     .when(F.col("precio") <= 30, "$15 a $30")
     .otherwise("Más de $30").alias("rango_precio"),
    "early_access")

dim_juego.write.mode("overwrite").parquet(f"{MODELO}/dim_juego")

verificacion = spark.read.parquet(f"{MODELO}/dim_juego")
print("dim_juego:", f"{verificacion.count():,}", "filas")
verificacion.groupBy("rango_precio").count().orderBy("count", ascending=False).show()

dim_juego: 32,132 filas
+------------+-----+
|rango_precio|count|
+------------+-----+
|    Hasta $5|14882|
|    $5 a $15|10030|
|   $15 a $30| 3002|
|      Gratis| 1900|
| Desconocido| 1385|
|  Más de $30|  933|
+------------+-----+



### 3.2. dim_usuario

Una fila por usuario, con la cantidad de juegos de su biblioteca. Se usa para contar jugadores distintos en las preguntas 1 y 5.

In [7]:
dim_usuario = usuarios.select("user_id", "cantidad_juegos")

dim_usuario.write.mode("overwrite").parquet(f"{MODELO}/dim_usuario")

verificacion = spark.read.parquet(f"{MODELO}/dim_usuario")
print("dim_usuario:", f"{verificacion.count():,}", "filas")
print("Usuarios con la biblioteca vacía:", f"{verificacion.filter(F.col('cantidad_juegos') == 0).count():,}")

dim_usuario: 87,626 filas
Usuarios con la biblioteca vacía: 16,714


En la exploración (sección 5 del notebook de limpieza) se contaron 16.806 bibliotecas vacías, pero ese conteo se hizo sobre las filas originales, antes de eliminar los usuarios repetidos. Para explicar la diferencia con los 16.714 usuarios de `dim_usuario`, contamos las filas y los usuarios distintos con biblioteca vacía en el archivo original.

In [8]:
crudo = spark.read.json("/lnd/steam/australian_users_items.json.gz")
vacios = crudo.filter(F.col("items_count") == 0)
print("Filas con biblioteca vacía:", f"{vacios.count():,}")
print("Usuarios distintos con biblioteca vacía:", f"{vacios.select('user_id').distinct().count():,}")

Filas con biblioteca vacía: 16,806
Usuarios distintos con biblioteca vacía: 16,714


### 3.3. dim_genero

Antes de construir la dimensión listamos todos los géneros del catálogo con la cantidad de juegos de cada uno, para clasificar los valores que no describen un tipo de juego.]

In [9]:
juego_genero.groupBy("genero").count().orderBy("count", ascending=False).show(50, truncate=False)

+---------------------+-----+
|genero               |count|
+---------------------+-----+
|Indie                |15858|
|Action               |11319|
|Casual               |8282 |
|Adventure            |8242 |
|Strategy             |6957 |
|Simulation           |6699 |
|RPG                  |5479 |
|Free to Play         |2031 |
|Early Access         |1462 |
|Sports               |1257 |
|Massively Multiplayer|1108 |
|Racing               |1083 |
|Design & Illustration|460  |
|Utilities            |340  |
|Web Publishing       |268  |
|Animation & Modeling |183  |
|Education            |125  |
|Video Production     |116  |
|Software Training    |105  |
|Audio Production     |93   |
|Photo Editing        |77   |
|Accounting           |7    |
+---------------------+-----+



La lista tiene 22 valores. Diez describen un tipo de juego. Free to Play y Early Access describen el modelo de negocio o el estado del juego, y los diez restantes son categorías de software. En lugar de eliminarlos, la dimensión agrega la columna `categoria` para que cada pregunta filtre los valores que corresponden. La relación entre juegos y géneros, que es de muchos a muchos, se guarda en la tabla puente `puente_juego_genero`.

In [10]:
modelo_negocio = ["Free to Play", "Early Access"]
software = ["Design & Illustration", "Utilities", "Web Publishing", "Animation & Modeling",
            "Education", "Video Production", "Software Training", "Audio Production",
            "Photo Editing", "Accounting"]

dim_genero = (juego_genero.select("genero").distinct()
    .withColumn("categoria",
        F.when(F.col("genero").isin(modelo_negocio), "Modelo de negocio")
         .when(F.col("genero").isin(software), "Software")
         .otherwise("Juego")))

puente_juego_genero = juego_genero.select("game_id", "genero")

dim_genero.write.mode("overwrite").parquet(f"{MODELO}/dim_genero")
puente_juego_genero.write.mode("overwrite").parquet(f"{MODELO}/puente_juego_genero")

print("puente_juego_genero:", f"{spark.read.parquet(f'{MODELO}/puente_juego_genero').count():,}", "filas")
spark.read.parquet(f"{MODELO}/dim_genero").orderBy("categoria", "genero").show(30, truncate=False)

puente_juego_genero: 71,551 filas
+---------------------+-----------------+
|genero               |categoria        |
+---------------------+-----------------+
|Action               |Juego            |
|Adventure            |Juego            |
|Casual               |Juego            |
|Indie                |Juego            |
|Massively Multiplayer|Juego            |
|RPG                  |Juego            |
|Racing               |Juego            |
|Simulation           |Juego            |
|Sports               |Juego            |
|Strategy             |Juego            |
|Early Access         |Modelo de negocio|
|Free to Play         |Modelo de negocio|
|Accounting           |Software         |
|Animation & Modeling |Software         |
|Audio Production     |Software         |
|Design & Illustration|Software         |
|Education            |Software         |
|Photo Editing        |Software         |
|Software Training    |Software         |
|Utilities            |Software         |


### 3.4. dim_desarrollador

La columna `developer` puede traer varios desarrolladores separados por coma. Antes de separarlos revisamos qué valores contienen comas, porque algunos nombres de empresa incluyen una coma propia (por ejemplo, "Inc." o "LLC").

In [11]:
print("Juegos sin desarrollador:", f"{juegos.filter(F.col('developer').isNull()).count():,}")
print("Juegos con coma en el desarrollador:", f"{juegos.filter(F.col('developer').contains(',')).count():,}")

fragmentos = (juegos.filter(F.col("developer").contains(","))
    .select(F.explode(F.split("developer", ",")).alias("fragmento"))
    .select(F.trim("fragmento").alias("fragmento")))

fragmentos.groupBy("fragmento").count().orderBy("count", ascending=False).show(25, truncate=False)

Juegos sin desarrollador: 3,298
Juegos con coma en el desarrollador: 4,352
+-------------------------+-----+
|fragmento                |count|
+-------------------------+-----+
|LLC                      |1002 |
|SmiteWorks USA           |813  |
|Inc.                     |419  |
|LTD.                     |336  |
|KOEI TECMO GAMES CO.     |304  |
|Feral Interactive (Mac)  |303  |
|Feral Interactive (Linux)|207  |
|Ltd.                     |162  |
|Relic Entertainment      |107  |
|Team NINJA               |72   |
|Monolith Productions     |66   |
|KADOKAWA                 |66   |
|Idea Factory             |63   |
|Capcom U.S.A.            |60   |
|Compile Heart            |57   |
|THQ Nordic               |55   |
|Raven Software           |52   |
|Inc                      |51   |
|Sledgehammer Games       |50   |
|Yoji Ojima               |45   |
|Gearbox Software         |40   |
|Aspyr (Mac & Linux)      |39   |
|QLOC                     |38   |
|TT Games                 |38   |
|Rainbo

El resultado muestra que separar en cada coma es incorrecto: los fragmentos más frecuentes son sufijos de razón social (LLC, Inc., LTD., Ltd.) y mitades de nombres como "KOEI TECMO GAMES CO.". Por eso separamos solo en las comas que no van seguidas de un sufijo de empresa, y verificamos que después de separar no quede ningún sufijo suelto.

In [13]:
sufijos = r"(?:LLC|L\.L\.C|Inc|LTD|Ltd|Limited|Co|Corp|Corporation|GmbH|S\.A|S\.L|SRL|S\.R\.L|Pty|AB|Oy|LLP|KG|BV|B\.V|s\.r\.o|SAS|AG|ApS|AS)"
separador = r",(?!\s*(?i)" + sufijos + r"\.?(?:\s|,|$))\s*"

desarrolladores = (juegos.filter(F.col("developer").isNotNull())
    .select("game_id", F.explode(F.split("developer", separador)).alias("desarrollador"))
    .select("game_id", F.trim("desarrollador").alias("desarrollador"))
    .filter(F.col("desarrollador") != ""))

solo_sufijo = desarrolladores.filter(F.col("desarrollador").rlike(r"(?i)^" + sufijos + r"\.?$"))
print("Fragmentos que son solo un sufijo:", solo_sufijo.count())

(desarrolladores.filter(F.col("game_id").isin(
        [r.game_id for r in juegos.filter(F.col("developer").contains(",")).select("game_id").collect()]))
    .groupBy("desarrollador").count().orderBy("count", ascending=False).show(25, truncate=False))

Fragmentos que son solo un sufijo: 0
+--------------------------+-----+
|desarrollador             |count|
+--------------------------+-----+
|SmiteWorks USA, LLC       |813  |
|KOEI TECMO GAMES CO., LTD.|304  |
|Feral Interactive (Mac)   |303  |
|Feral Interactive (Linux) |207  |
|Relic Entertainment       |107  |
|Team NINJA                |72   |
|KADOKAWA                  |66   |
|Monolith Productions, Inc.|65   |
|Idea Factory              |61   |
|Compile Heart             |57   |
|THQ Nordic                |55   |
|Raven Software            |52   |
|Capcom U.S.A., Inc.       |51   |
|Sledgehammer Games        |50   |
|Yoji Ojima                |45   |
|Aspyr (Mac & Linux)       |39   |
|Gearbox Software          |39   |
|TT Games                  |38   |
|QLOC                      |38   |
|Rainbow Studios           |36   |
|CAPCOM Co., Ltd.          |35   |
|Traveller's Tales         |35   |
|NetherRealm Studios       |31   |
|Sting                     |30   |
|Crystal Dynamics 

Con este criterio no queda ningún sufijo suelto y los nombres de empresa se conservan completos (por ejemplo, "SmiteWorks USA, LLC" o "KOEI TECMO GAMES CO., LTD."). Guardamos la dimensión con los desarrolladores distintos y la tabla puente que une cada juego con sus desarrolladores. Los 3.298 juegos sin desarrollador no tienen filas en el puente, por lo que quedan fuera de la pregunta 5.

In [14]:
puente_juego_desarrollador = desarrolladores.dropDuplicates(["game_id", "desarrollador"])
dim_desarrollador = puente_juego_desarrollador.select("desarrollador").distinct()

dim_desarrollador.write.mode("overwrite").parquet(f"{MODELO}/dim_desarrollador")
puente_juego_desarrollador.write.mode("overwrite").parquet(f"{MODELO}/puente_juego_desarrollador")

puente = spark.read.parquet(f"{MODELO}/puente_juego_desarrollador")
print("dim_desarrollador:", f"{spark.read.parquet(f'{MODELO}/dim_desarrollador').count():,}", "desarrolladores")
print("puente_juego_desarrollador:", f"{puente.count():,}", "filas")
print("Juegos con más de un desarrollador:",
      f"{puente.groupBy('game_id').count().filter(F.col('count') > 1).count():,}")

dim_desarrollador: 11,394 desarrolladores
puente_juego_desarrollador: 32,205 filas
Juegos con más de un desarrollador: 2,458


## 4. Hechos

### 4.1. hecho_biblioteca

Una fila por cada juego en la biblioteca de un usuario, con los minutos jugados. Se agregan dos columnas calculadas: `horas_totales`, para presentar los resultados en horas, y `jugado`, que indica si el juego se abrió alguna vez (pregunta 4). También se marca con `en_catalogo` si el juego existe en `dim_juego`: las filas de juegos fuera del catálogo se conservan, pero las preguntas que agrupan por características del juego solo pueden usar las que tienen esa marca.

In [15]:
ids_catalogo = dim_juego.select("game_id").withColumn("en_catalogo", F.lit(True))

hecho_biblioteca = (usuario_juego
    .join(ids_catalogo, "game_id", "left")
    .select(
        "user_id",
        "game_id",
        "minutos_totales",
        "minutos_2semanas",
        F.round(F.col("minutos_totales") / 60, 2).alias("horas_totales"),
        (F.col("minutos_totales") > 0).alias("jugado"),
        F.coalesce(F.col("en_catalogo"), F.lit(False)).alias("en_catalogo")))

hecho_biblioteca.write.mode("overwrite").parquet(f"{MODELO}/hecho_biblioteca")

verificacion = spark.read.parquet(f"{MODELO}/hecho_biblioteca")
print("hecho_biblioteca:", f"{verificacion.count():,}", "filas")
print("Filas de juegos fuera del catálogo:", f"{verificacion.filter(~F.col('en_catalogo')).count():,}")
print("Juegos nunca abiertos:", f"{verificacion.filter(~F.col('jugado')).count():,}")

hecho_biblioteca: 5,094,082 filas
Filas de juegos fuera del catálogo: 849,270
Juegos nunca abiertos: 1,847,730


### 4.2. hecho_resena

Una fila por reseña, con la fecha y si el usuario recomienda el juego. Responde la pregunta 3. El texto de la reseña no se incluye porque ninguna pregunta lo analiza; se conserva en la zona refinada. Igual que en la biblioteca, se marca con `en_catalogo` si el juego existe en `dim_juego`.

In [16]:
hecho_resena = (resenas
    .join(ids_catalogo, "game_id", "left")
    .select(
        "user_id",
        "game_id",
        "fecha_publicacion",
        "recomienda",
        F.coalesce(F.col("en_catalogo"), F.lit(False)).alias("en_catalogo")))

hecho_resena.write.mode("overwrite").parquet(f"{MODELO}/hecho_resena")

verificacion = spark.read.parquet(f"{MODELO}/hecho_resena")
print("hecho_resena:", f"{verificacion.count():,}", "filas")
print("Reseñas de juegos fuera del catálogo:", f"{verificacion.filter(~F.col('en_catalogo')).count():,}")
print("Reseñas que recomiendan:", f"{verificacion.filter(F.col('recomienda')).count():,}")

hecho_resena: 58,431 filas
Reseñas de juegos fuera del catálogo: 5,234
Reseñas que recomiendan: 51,692


## 5. Tablas Hive

Creamos la base `steam` en Hive y, sobre cada carpeta de `/models/steam`, una tabla externa en formato Parquet. Al ser externas, Hive solo guarda la definición de la tabla (nombre, columnas y ubicación); los datos siguen en HDFS, y si una tabla se elimina, sus archivos se conservan.

La base y las tablas se crean con la ruta completa de HDFS (`hdfs://namenode:8020/...`). El metastore de Hive de este entorno tiene configurado el NameNode en el puerto 9000, mientras que el NameNode atiende en el 8020; indicando la ruta completa, Hive usa el puerto correcto. La base se ubica en `/models/steam/steam.db`, separada de las carpetas de las tablas.

In [21]:
spark.sql("CREATE DATABASE IF NOT EXISTS steam LOCATION 'hdfs://namenode:8020/models/steam/steam.db'")
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
| big_data|
|  default|
|    steam|
+---------+



In [22]:
RUTA_HDFS = "hdfs://namenode:8020" + MODELO

tablas = {
    "dim_juego": "game_id STRING, nombre STRING, publisher STRING, anio_lanzamiento INT, precio DOUBLE, es_gratuito BOOLEAN, rango_precio STRING, early_access BOOLEAN",
    "dim_usuario": "user_id STRING, cantidad_juegos BIGINT",
    "dim_genero": "genero STRING, categoria STRING",
    "puente_juego_genero": "game_id STRING, genero STRING",
    "dim_desarrollador": "desarrollador STRING",
    "puente_juego_desarrollador": "game_id STRING, desarrollador STRING",
    "hecho_biblioteca": "user_id STRING, game_id STRING, minutos_totales BIGINT, minutos_2semanas BIGINT, horas_totales DOUBLE, jugado BOOLEAN, en_catalogo BOOLEAN",
    "hecho_resena": "user_id STRING, game_id STRING, fecha_publicacion DATE, recomienda BOOLEAN, en_catalogo BOOLEAN",
}

for nombre, columnas in tablas.items():
    spark.sql(f"DROP TABLE IF EXISTS steam.{nombre}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE steam.{nombre} ({columnas})
        STORED AS PARQUET
        LOCATION '{RUTA_HDFS}/{nombre}'
    """)
    print(f"Tabla steam.{nombre} creada")

Tabla steam.dim_juego creada
Tabla steam.dim_usuario creada
Tabla steam.dim_genero creada
Tabla steam.puente_juego_genero creada
Tabla steam.dim_desarrollador creada
Tabla steam.puente_juego_desarrollador creada
Tabla steam.hecho_biblioteca creada
Tabla steam.hecho_resena creada


## 6. Verificación

Comprobamos que Hive lee los datos del modelo: listamos las tablas de la base `steam` y contamos las filas de cada una con SQL, consultando las tablas Hive y no los archivos. Los conteos deben coincidir con los de las secciones 3 y 4.

In [23]:
spark.sql("SHOW TABLES IN steam").show(truncate=False)

conteos = [(nombre, spark.sql(f"SELECT COUNT(*) AS n FROM steam.{nombre}").first()["n"]) for nombre in tablas]
spark.createDataFrame(conteos, ["tabla", "filas"]).show(truncate=False)

+---------+--------------------------+-----------+
|namespace|tableName                 |isTemporary|
+---------+--------------------------+-----------+
|steam    |dim_desarrollador         |false      |
|steam    |dim_genero                |false      |
|steam    |dim_juego                 |false      |
|steam    |dim_usuario               |false      |
|steam    |hecho_biblioteca          |false      |
|steam    |hecho_resena              |false      |
|steam    |puente_juego_desarrollador|false      |
|steam    |puente_juego_genero       |false      |
+---------+--------------------------+-----------+

+--------------------------+-------+
|tabla                     |filas  |
+--------------------------+-------+
|dim_juego                 |32132  |
|dim_usuario               |87626  |
|dim_genero                |22     |
|puente_juego_genero       |71551  |
|dim_desarrollador         |11394  |
|puente_juego_desarrollador|32205  |
|hecho_biblioteca          |5094082|
|hecho_resena    

### Resumen del modelo

El modelo estrella quedó guardado en Parquet en `/models/steam` y registrado como tablas externas en la base `steam` de Hive:

| Tabla | Tipo | Filas | Uso |
|---|---|---|---|
| `hecho_biblioteca` | Hecho | 5.094.082 | Minutos jugados por usuario y juego (preguntas 1, 2, 4 y 5) |
| `hecho_resena` | Hecho | 58.431 | Recomendación de cada reseña (pregunta 3) |
| `dim_juego` | Dimensión | 32.132 | Atributos del juego, incluido el rango de precio |
| `dim_usuario` | Dimensión | 87.626 | Usuarios y cantidad de juegos |
| `dim_genero` | Dimensión | 22 | Géneros con su categoría (juego, modelo de negocio o software) |
| `dim_desarrollador` | Dimensión | 11.394 | Desarrolladores distintos |
| `puente_juego_genero` | Puente | 71.551 | Relación de muchos a muchos entre juegos y géneros |
| `puente_juego_desarrollador` | Puente | 32.205 | Relación de muchos a muchos entre juegos y desarrolladores |

Las preguntas se responden en un notebook aparte, consultando estas tablas Hive.